# How much of a protein crystal is solvent

**Data kit lab** `pdb-1hsg-matthews-solvent` · Discovery · about 20 minutes · beginner

**The question.** How tightly is the protein packed in the 1HSG crystal? What are its Matthews coefficient and solvent content?

**What you do.** You compute two numbers from a deposited crystal structure and check them against the database entry.

1. Open the structure file and find the unit cell.
2. Compute the volume of the unit cell.
3. Count the protein chains in the cell.
4. Look up the mass of one chain.
5. Compute the Matthews coefficient and the solvent content.
6. Compare your numbers with the ones the RCSB PDB entry reports.

**The data.** PDB entry 1HSG: HIV-1 protease with the inhibitor L-735,524, solved by X-ray crystallography. It comes from RCSB PDB, a data centre of the public PDB archive. The licence is CC0 1.0 (public domain). Nothing is copied into this notebook: the first cell downloads two small files from RCSB PDB.

**Where this data lives.** Public structures live in the PDB archive. Depositors submit them and each one is reviewed before release. RCSB PDB names pharmaceutical and biotechnology companies among the users of the archive. Where a company keeps its own unpublished structures is not covered by a source this lab has opened.

**How to run.** Run the cells from top to bottom. You need Python 3 and nothing else.

In [ ]:
# datakit:setup (generated from lab.yaml and expected.json by scripts/datakit/harness.py sync; do not edit by hand)
import hashlib, json, os, pathlib, urllib.request

LAB = 'pdb-1hsg-matthews-solvent'
FILES = json.loads(r'''{
 "structure": {
  "url": "https://files.rcsb.org/download/1HSG.pdb",
  "name": "structure.pdb",
  "sha256": "d430675c358060808ef614002cfdcfcfa824714d89f1240adf5749d2ba60fed2",
  "bytes": 168075
 },
 "entity": {
  "url": "https://data.rcsb.org/rest/v1/core/polymer_entity/1HSG/1",
  "name": "entity",
  "sha256": "727cb3b1ea1abb960096fe65cca63f55116f5415a21d97b7431d07043c76f682",
  "bytes": 48535
 }
}''')
PUBLISHED = json.loads(r'''{
 "matthews_coefficient": {
  "label": "Matthews coefficient",
  "value": 2.8,
  "tolerance": {
   "abs": 0.05
  }
 },
 "solvent_content_percent": {
  "label": "Solvent content",
  "value": 56.1,
  "tolerance": {
   "abs": 0.05
  }
 }
}''')
PUBLISHED_FROM = 'RCSB Data API record of entry 1HSG, field exptl_crystal (density_Matthews, density_percent_sol) | https://data.rcsb.org/rest/v1/core/entry/1HSG'
DATA_DIR = pathlib.Path(os.environ.get('DATAKIT_DATA', 'datakit-data/' + LAB))


def data_file(file_id):
    """Path of one dataset file. Downloads it from its source on first use and checks the checksum."""
    f = FILES[file_id]
    path = DATA_DIR / f['name']
    if not path.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print('downloading', f['url'])
        urllib.request.urlretrieve(f['url'], path)
    if f['sha256']:
        got = hashlib.sha256(path.read_bytes()).hexdigest()
        if got != f['sha256']:
            print('WARNING:', path.name, 'differs from the copy this lab was checked against (the source file changed)')
    return path


def within(computed, check):
    tol, pub = check['tolerance'], check['value']
    if 'exact' in tol:
        return computed == pub
    limit = tol['abs'] if 'abs' in tol else tol['rel'] * abs(pub)
    return abs(computed - pub) <= limit + 1e-12


def report(results):
    """Print computed against published values and save the computed values for the harness."""
    print('Published values:', PUBLISHED_FROM)
    print(f"{'check':<44}{'published':>12}{'computed':>14}  result")
    for key, check in PUBLISHED.items():
        got = results.get(key)
        shown = '(missing)' if got is None else (f'{got:.4f}' if isinstance(got, float) else str(got))
        verdict = 'missing' if got is None else ('match' if within(got, check) else 'DIFFERENT')
        print(f"{check['label'][:43]:<44}{str(check['value']):>12}{shown:>14}  {verdict}")
    out = pathlib.Path(os.environ.get('DATAKIT_RESULTS', 'results.json'))
    out.write_text(json.dumps(results, indent=1))
    print('saved', out)


## 1. Open the structure file

The file is in the PDB text format. Each line is one record. The first six characters name the record type.

- `CRYST1` holds the unit cell of the crystal.
- `ATOM` lines hold the protein atoms. `HETATM` lines hold everything else: here the inhibitor and water.
- `REMARK` lines hold notes, some of them in a fixed layout.

In [ ]:
from collections import Counter

lines = data_file("structure").read_text().splitlines()
print("lines in the file:", len(lines))
for line in lines:
    if line.startswith(("HEADER", "TITLE", "EXPDTA")):
        print(line.rstrip())

kinds = Counter(line[:6].strip() for line in lines)
print({kind: kinds[kind] for kind in ("CRYST1", "SEQRES", "ATOM", "HETATM", "REMARK")})

## 2. The unit cell

A crystal is one small box, the unit cell, repeated in all three directions. The `CRYST1` line gives the three edge lengths (a, b, c, in angstrom), the three angles (in degrees), the space group and a count called Z.

Print the line, then try the obvious way to read it: split on spaces.

In [ ]:
cryst1 = next(line for line in lines if line.startswith("CRYST1"))
print(cryst1.rstrip())
pieces = cryst1.split()
print(len(pieces), "pieces:", pieces)

**What goes wrong here.** The space group is written `P 21 21 2`, with spaces inside it. Splitting on spaces breaks it into four pieces, so the number of pieces changes from one space group to the next.

PDB-format lines are cut by column position, not by spaces. The slices below were checked against the cell values in the entry's database record (59.57, 87.07, 46.71, three angles of 90, Z of 8).

In [ ]:
a = float(cryst1[6:15])
b = float(cryst1[15:24])
c = float(cryst1[24:33])
alpha = float(cryst1[33:40])
beta = float(cryst1[40:47])
gamma = float(cryst1[47:54])
space_group = cryst1[55:66].strip()
z_on_file = int(cryst1[66:70])

print("edges (angstrom):", a, b, c)
print("angles (degrees):", alpha, beta, gamma)
print("space group:", space_group)
print("Z:", z_on_file)

## 3. The volume of the unit cell

When all three angles are 90 degrees the box is a brick and the volume is a × b × c. The formula below also works for tilted cells, so you can reuse it on other entries.

In [ ]:
import math

cos_a, cos_b, cos_g = (math.cos(math.radians(angle)) for angle in (alpha, beta, gamma))
tilt = math.sqrt(1 - cos_a**2 - cos_b**2 - cos_g**2 + 2 * cos_a * cos_b * cos_g)
cell_volume = a * b * c * tilt

print("tilt factor:", round(tilt, 6), "(1 means a brick-shaped cell)")
print("unit-cell volume:", round(cell_volume, 1), "cubic angstrom")

## 4. How many protein chains are in the cell?

The file lists only one part of the cell, the asymmetric unit. The symmetry operators of the space group copy that part to fill the rest of the cell. They are listed in `REMARK 290`.

So: chains in the cell = chains listed in the file × number of symmetry operators.

In [ ]:
chains = sorted({line[21] for line in lines if line.startswith("ATOM")})
operators = sorted({int(line[19:23]) for line in lines if line.startswith("REMARK 290   SMTRY")})

for line in lines:
    if line.startswith("REMARK 290") and line[10:22].strip().endswith("555"):
        print(line.rstrip())

chains_in_cell = len(chains) * len(operators)
print("protein chains in the file:", chains)
print("symmetry operators:", len(operators))
print("chains in the cell:", chains_in_cell, "| Z on the CRYST1 line:", z_on_file)
assert chains_in_cell == z_on_file, "the count from the file and Z disagree"

The two counts agree: Z is 8. RCSB PDB defines this Z as the number of polymer chains in the unit cell. It is not the number of protease dimers (that would be 4).

The file also holds atoms that are not protein. Look at what they are. They are **not** counted in the mass in the next step.

In [ ]:
other = {}
for line in lines:
    if line.startswith("HETATM"):
        name = line[17:20]
        residue = (line[21], line[22:26])
        other.setdefault(name, set()).add(residue)
for name, residues in other.items():
    print(name, "molecules in the file:", len(residues))

## 5. The mass of one chain

The second file is the database record of the protein in this entry. It gives the formula mass of one chain.

**What goes wrong here.** The record gives the mass in kilodaltons. The Matthews coefficient is per dalton. Convert before you divide.

In [ ]:
import json

entity = json.loads(data_file("entity").read_text())
name = entity["rcsb_polymer_entity"]["pdbx_description"]
mass_kda = entity["rcsb_polymer_entity"]["formula_weight"]
length = entity["entity_poly"]["rcsb_sample_sequence_length"]
chain_mass = mass_kda * 1000  # kilodaltons to daltons

seqres = {}
for line in lines:
    if line.startswith("SEQRES"):
        seqres.setdefault(line[11], []).extend(line[19:].split())

print("protein:", name, "| chains:", entity["entity_poly"]["pdbx_strand_id"])
print("mass of one chain:", mass_kda, "kDa =", chain_mass, "Da")
print("residues per chain, database record:", length)
print("residues per chain, SEQRES lines:", {chain: len(residues) for chain, residues in seqres.items()})
assert all(len(residues) == length for residues in seqres.values()), "sequence length differs between the two files"

## 6. The Matthews coefficient

The Matthews coefficient is the crystal volume available per dalton of protein:

**V_M = cell volume / (Z × mass of one chain)**

Its unit is cubic angstrom per dalton. A larger value means looser packing and more room for solvent.

In [ ]:
protein_mass_in_cell = z_on_file * chain_mass
matthews = cell_volume / protein_mass_in_cell

print("protein mass in the cell:", protein_mass_in_cell, "Da")
print("Matthews coefficient:", round(matthews, 4), "cubic angstrom per dalton")

## 7. The solvent content

RCSB PDB's definition of the solvent field gives this formula:

**solvent fraction = 1 − 1.23 / V_M**

The 1.23 is the volume, in cubic angstrom, that one dalton of protein takes up. It comes from an assumed protein density: a partial specific volume of 0.74 cm³ per gram. So 1.23 / V_M is the share of the cell filled by protein, and the rest is solvent.

This is an estimate from an assumed value, not a measurement.

In [ ]:
PROTEIN_VOLUME_PER_DALTON = 1.23  # cubic angstrom per dalton, the constant in the field definition

protein_fraction = PROTEIN_VOLUME_PER_DALTON / matthews
solvent_percent = 100 * (1 - protein_fraction)

print("share of the cell filled by protein:", round(100 * protein_fraction, 2), "%")
print("solvent content:", round(solvent_percent, 2), "%")

## 8. Check against the database entry

`PUBLISHED` holds the two values copied by code from the entry's record at RCSB PDB (field `exptl_crystal`). `report` prints your value beside each one and says whether it is inside the tolerance: half a unit in the last digit the record prints.

In [ ]:
RESULTS = {
    "matthews_coefficient": float(matthews),
    "solvent_content_percent": float(solvent_percent),
}
report(RESULTS)

## 9. How far can you trust the second decimal?

The database record prints 2.8 and 56.1. The structure file you downloaded prints the same two values with one more digit, in `REMARK 280`. Compare at that precision.

In [ ]:
for line in lines:
    if line.startswith("REMARK 280") and ("MATTHEWS" in line or "SOLVENT CONTENT" in line):
        print(line.rstrip())
print()
print(f"your values to two decimals: {matthews:.2f} and {solvent_percent:.2f}")

exact_constant = 0.74 * 1e24 / 6.02e23  # the calculation the field definition gives for 1.23
print()
print("constant before rounding:", round(exact_constant, 4))
print("solvent content with 1.23 and your V_M:        ", round(100 * (1 - 1.23 / matthews), 2))
print("solvent content with the unrounded constant:    ", round(100 * (1 - exact_constant / matthews), 2))
print("solvent content with 1.23 and V_M rounded to 2.80:", round(100 * (1 - 1.23 / 2.80), 2))

The Matthews coefficient agrees to two decimals. The solvent content does not: you get 56.12 and the file says 56.10.

Three equally reasonable ways of doing the same sum give 56.07, 56.12 and 56.15. The entry does not record which mass or which constant was used when the value was first calculated. The honest reading: the solvent content of this crystal is about 56 %, and the second decimal carries no information.

## 10. Two mistakes worth making once

Run the cell and see what each mistake does to the answer.

In [ ]:
asymmetric_unit_volume = cell_volume / len(operators)
print(f"asymmetric-unit volume divided by ONE chain: {asymmetric_unit_volume / chain_mass:.2f}")
print(f"asymmetric-unit volume divided by its {len(chains)} chains: {asymmetric_unit_volume / (len(chains) * chain_mass):.2f}")
print("mass left in kilodaltons:", round(cell_volume / (z_on_file * mass_kda), 1))

- The field definition speaks of the volume of the asymmetric unit over the mass of a monomer. Here the asymmetric unit holds two chains, so that reading doubles the answer. Always divide a volume by all the protein inside that same volume.
- A mass left in kilodaltons gives a value 1,000 times too large.

## What the result says

A little over half of this crystal is solvent, not protein. Your two numbers match the database entry at the precision the entry's record prints.

## Try next

1. Count the inhibitor in the mass as well. Its formula is on the `FORMUL` line for `MK1`. How far does the coefficient move, and which version does the entry's value agree with?
2. Change `1HSG` in the two file addresses of the first cell to another crystal entry and rerun. Does chains × operators still equal Z?
3. Work out the mass of one chain yourself from the `SEQRES` lines and a table of residue masses. How close do you get to the database record?